In [1]:
from analisis_exploratorio import funciones as fn
import pandas as pd
import numpy as np
from typing import Dict, Any, Optional

In [2]:
df_crudo = fn.cargar_datos()
dic_atributos = fn.diccionario_tipos_atributos()
atributos_a_eliminar = np.concatenate([dic_atributos['atributos_min_max_avg_terceros'], ["url", "timedelta"]])
df_crudo.drop(atributos_a_eliminar, axis=1, inplace=True)


In [3]:
# python
def dataframe_boxplot(df: pd.DataFrame,
                     clamp_inferior_a_cero: bool = True,
                     incluir_no_numericos: bool = False) -> pd.DataFrame:
    """
    Calcula métricas tipo boxplot para cada columna y las devuelve en un DataFrame.
    """
    filas = []
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            if not incluir_no_numericos:
                continue
            filas.append({
                "atributo": col,
                "q1": None, "q2": None, "q3": None, "iqr": None,
                "limite_inferior": None, "limite_superior": None,
                "minimo": None, "maximo": None,
                "n_valores": int(df[col].shape[0] - df[col].isna().sum()),
                "n_nulos": int(df[col].isna().sum())
            })
            continue

        s = df[col].dropna()
        n_valores = int(s.shape[0])
        n_nulos = int(df[col].isna().sum())

        if n_valores == 0:
            filas.append({
                "atributo": col,
                "q1": None, "q2": None, "q3": None, "iqr": None,
                "limite_inferior": None, "limite_superior": None,
                "minimo": None, "maximo": None,
                "n_valores": n_valores, "n_nulos": n_nulos
            })
            continue

        q1 = float(s.quantile(0.25))
        q2 = float(s.quantile(0.50))
        q3 = float(s.quantile(0.75))
        iqr = float(q3 - q1)
        limite_inferior = float(q1 - 1.5 * iqr)
        if clamp_inferior_a_cero and limite_inferior < 0:
            limite_inferior = 0.0
        limite_superior = float(q3 + 1.5 * iqr)
        minimo = float(s.min())
        maximo = float(s.max())

        filas.append({
            "atributo": col,
            "q1": q1, "q2": q2, "q3": q3, "iqr": iqr,
            "limite_inferior": limite_inferior, "limite_superior": limite_superior,
            "minimo": minimo, "maximo": maximo,
            "n_valores": n_valores, "n_nulos": n_nulos
        })

    return pd.DataFrame(filas)

In [4]:
datos_bolxplots = dataframe_boxplot(df_crudo, clamp_inferior_a_cero=False)
datos_bolxplots

,atributo,q1,q2,q3,iqr,limite_inferior,limite_superior,minimo,maximo,n_valores,n_nulos
0,n_tokens_title,9.000000,10.000000,12.000000,3.000000e+00,4.500000e+00,16.500000,2.00000,23.000000,39644,0
1,n_tokens_content,246.000000,409.000000,716.000000,4.700000e+02,-4.590000e+02,1421.000000,0.00000,8474.000000,39644,0
2,n_unique_tokens,0.470870,0.539226,0.608696,1.378252e-01,2.641326e-01,0.815433,0.00000,701.000000,39644,0
3,n_non_stop_words,1.000000,1.000000,1.000000,4.314000e-09,1.000000e+00,1.000000,0.00000,1042.000000,39644,0
4,n_non_stop_unique_tokens,0.625739,0.690476,0.754630,1.288902e-01,4.324041e-01,0.947965,0.00000,650.000000,39644,0
5,num_hrefs,4.000000,8.000000,14.000000,1.000000e+01,-1.100000e+01,29.000000,0.00000,304.000000,39644,0
6,num_self_hrefs,1.000000,3.000000,4.000000,3.000000e+00,-3.500000e+00,8.500000,0.00000,116.000000,39644,0
7,num_imgs,1.000000,1.000000,4.000000,3.000000e+00,-3.500000e+00,8.500000,0.00000,128.000000,39644,0
8,num_videos,0.000000,0.000000,1.000000,1.000000e+00,-1.500000e+00,2.500000,0.00000,91.000000,39644,0
9,average_token_length,4.478404,4.664082,4.854839,3.764347e-01,3.913752e+00,5.419491,0.00000,8.041534,39644,0


In [5]:
# python
def limpiar_por_limites_boxplot(df: pd.DataFrame,
                                df_boxplot: pd.DataFrame,
                                atributos: list) -> pd.DataFrame:
    """
    Filtra el DataFrame usando los límites inferior y superior de los atributos indicados,
    según el DataFrame de boxplot.
    """
    df_limpio = df.copy()
    for atributo in atributos:
        if atributo not in df_boxplot["atributo"].values:
            raise ValueError(f"El atributo `{atributo}` no está en el DataFrame de boxplot.")
        limites = df_boxplot[df_boxplot["atributo"] == atributo].iloc[0]
        lim_inf = limites["limite_inferior"]
        lim_sup = limites["limite_superior"]
        df_limpio = df_limpio[(df_limpio[atributo] >= lim_inf) & (df_limpio[atributo] <= lim_sup)]
    df_limpio = df_limpio.reset_index(drop=True)
    return df_limpio

In [6]:
df_crudo

,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,num_videos,average_token_length,...,min_positive_polarity,max_positive_polarity,avg_negative_polarity,min_negative_polarity,max_negative_polarity,title_subjectivity,title_sentiment_polarity,abs_title_subjectivity,abs_title_sentiment_polarity,shares
0,12.0,219.0,0.663594,1.0,0.815385,4.0,2.0,1.0,0.0,4.680365,...,0.100000,0.70,-0.350000,-0.600,-0.200000,0.500000,-0.187500,0.000000,0.187500,593
1,9.0,255.0,0.604743,1.0,0.791946,3.0,1.0,1.0,0.0,4.913725,...,0.033333,0.70,-0.118750,-0.125,-0.100000,0.000000,0.000000,0.500000,0.000000,711
2,9.0,211.0,0.575130,1.0,0.663866,3.0,1.0,1.0,0.0,4.393365,...,0.100000,1.00,-0.466667,-0.800,-0.133333,0.000000,0.000000,0.500000,0.000000,1500
3,9.0,531.0,0.503788,1.0,0.665635,9.0,0.0,1.0,0.0,4.404896,...,0.136364,0.80,-0.369697,-0.600,-0.166667,0.000000,0.000000,0.500000,0.000000,1200
4,13.0,1072.0,0.415646,1.0,0.540890,19.0,19.0,20.0,0.0,4.682836,...,0.033333,1.00,-0.220192,-0.500,-0.050000,0.454545,0.136364,0.045455,0.136364,505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39639,11.0,346.0,0.529052,1.0,0.684783,9.0,7.0,1.0,1.0,4.523121,...,0.100000,0.75,-0.260000,-0.500,-0.125000,0.100000,0.000000,0.400000,0.000000,1800
39640,12.0,328.0,0.696296,1.0,0.885057,9.0,7.0,3.0,48.0,4.405488,...,0.136364,0.70,-0.211111,-0.400,-0.100000,0.300000,1.000000,0.200000,1.000000,1900
39641,10.0,442.0,0.516355,1.0,0.644128,24.0,1.0,12.0,1.0,5.076923,...,0.136364,0.50,-0.356439,-0.800,-0.166667,0.454545,0.136364,0.045455,0.136364,1900
39642,6.0,682.0,0.539493,1.0,0.692661,10.0,1.0,1.0,0.0,4.975073,...,0.062500,0.50,-0.205246,-0.500,-0.012500,0.000000,0.000000,0.500000,0.000000,1100


In [7]:
atributos_filtro = ["n_tokens_title", "n_tokens_content", "n_unique_tokens", "n_non_stop_unique_tokens", "num_hrefs", "num_self_hrefs", "average_token_length", "global_subjectivity", "global_sentiment_polarity", "global_rate_positive_words", "global_rate_negative_words", "rate_positive_words", "rate_negative_words", "shares"]
df_limpio = limpiar_por_limites_boxplot(
    df=df_crudo,
    df_boxplot=datos_bolxplots,
    atributos=atributos_filtro
)
df_limpio

,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,num_videos,average_token_length,...,min_positive_polarity,max_positive_polarity,avg_negative_polarity,min_negative_polarity,max_negative_polarity,title_subjectivity,title_sentiment_polarity,abs_title_subjectivity,abs_title_sentiment_polarity,shares
0,12.0,219.0,0.663594,1.0,0.815385,4.0,2.0,1.0,0.0,4.680365,...,0.100000,0.70,-0.350000,-0.600,-0.200000,0.500000,-0.187500,0.000000,0.187500,593
1,9.0,255.0,0.604743,1.0,0.791946,3.0,1.0,1.0,0.0,4.913725,...,0.033333,0.70,-0.118750,-0.125,-0.100000,0.000000,0.000000,0.500000,0.000000,711
2,9.0,531.0,0.503788,1.0,0.665635,9.0,0.0,1.0,0.0,4.404896,...,0.136364,0.80,-0.369697,-0.600,-0.166667,0.000000,0.000000,0.500000,0.000000,1200
3,10.0,370.0,0.559889,1.0,0.698198,2.0,2.0,0.0,0.0,4.359459,...,0.136364,0.60,-0.195000,-0.400,-0.100000,0.642857,0.214286,0.142857,0.214286,855
4,11.0,97.0,0.670103,1.0,0.836735,2.0,0.0,0.0,0.0,4.855670,...,0.400000,0.80,-0.125000,-0.125,-0.125000,0.125000,0.000000,0.375000,0.000000,3600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26734,11.0,346.0,0.529052,1.0,0.684783,9.0,7.0,1.0,1.0,4.523121,...,0.100000,0.75,-0.260000,-0.500,-0.125000,0.100000,0.000000,0.400000,0.000000,1800
26735,12.0,328.0,0.696296,1.0,0.885057,9.0,7.0,3.0,48.0,4.405488,...,0.136364,0.70,-0.211111,-0.400,-0.100000,0.300000,1.000000,0.200000,1.000000,1900
26736,10.0,442.0,0.516355,1.0,0.644128,24.0,1.0,12.0,1.0,5.076923,...,0.136364,0.50,-0.356439,-0.800,-0.166667,0.454545,0.136364,0.045455,0.136364,1900
26737,6.0,682.0,0.539493,1.0,0.692661,10.0,1.0,1.0,0.0,4.975073,...,0.062500,0.50,-0.205246,-0.500,-0.012500,0.000000,0.000000,0.500000,0.000000,1100


In [8]:

df_limpio.to_csv('dataset/datos_limpios_nuevo.csv', index=False)